In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

# Define your project directory path
PROJECT_PATH = '/content/drive/MyDrive/Implied-Volatility-project'

# Change the current working directory to your project folder
os.chdir(PROJECT_PATH)

# Verify you are in the right place
print("Current Working Directory:", os.getcwd())

Current Working Directory: /content/drive/MyDrive/Implied-Volatility-project


In [ ]:
import re
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ── 0. CONFIGURATION ─────────────────────────────────────────────────────────

DATA_PATH        = "data/dataset.csv"       # ← Reviewer: update path to your dataset location
DATETIME_COL     = "datetime"          # row-identifier: timestamp column
UNDERLYING_COL   = "underlying_price"  # underlying Nifty50 spot price
ID_COLS          = [DATETIME_COL, UNDERLYING_COL]  # cols to keep as-is during melt
SYMBOL_COL       = "symbol"            # name for the melted option-symbol column
TARGET_COL       = "iv"                # name for the melted IV value column

# Outlier capping per timestamp cross-section
LOWER_QUANTILE   = 0.01
UPPER_QUANTILE   = 0.99

# Output artefact paths
CLEANED_OBS_PATH = "cleaned_observed.parquet"
PREDICT_PATH     = "predict_template.parquet"
CS_STATS_PATH    = "cross_sectional_stats.parquet"

In [ ]:
# ── 1. LOAD RAW WIDE DATA ────────────────────────────────────────────────────

def load_wide_data(path: str) -> pd.DataFrame:
    """Load the raw wide-format CSV and print a structural summary."""
    df = pd.read_csv(path)

    option_cols = [c for c in df.columns if c not in ID_COLS]

    print(f"[LOAD] Shape            : {df.shape}")
    print(f"[LOAD] ID columns       : {ID_COLS}")
    print(f"[LOAD] Option columns   : {len(option_cols)}")
    print(f"[LOAD] Sample symbols   : {option_cols[:5]}")
    print(f"[LOAD] Total cells      : {df[option_cols].size:,}")
    print(f"[LOAD] Missing IV cells : {df[option_cols].isna().sum().sum():,} "
          f"({df[option_cols].isna().sum().sum() / df[option_cols].size * 100:.1f}%)")
    return df


raw_wide_df = load_wide_data(DATA_PATH)

[LOAD] Shape            : (975, 30)
[LOAD] ID columns       : ['datetime', 'underlying_price']
[LOAD] Option columns   : 28
[LOAD] Sample symbols   : ['NIFTY27JAN2625200CE', 'NIFTY27JAN2625300CE', 'NIFTY27JAN2625400CE', 'NIFTY27JAN2625500CE', 'NIFTY27JAN2625600CE']
[LOAD] Total cells      : 27,300
[LOAD] Missing IV cells : 5,460 (20.0%)


In [ ]:
# ── 2. WIDE → LONG RESHAPE WITH pd.melt() ────────────────────────────────────

def melt_to_long(df: pd.DataFrame) -> pd.DataFrame:
    """
    Melt all option columns into a single long-format DataFrame.

    Input  (wide): one row per timestamp, one column per option symbol
    Output (long): one row per (timestamp × option symbol), with columns
                   [datetime, underlying_price, symbol, iv]
    """
    option_cols = [c for c in df.columns if c not in ID_COLS]

    long_df = pd.melt(
        df,
        id_vars    = ID_COLS,
        value_vars = option_cols,
        var_name   = SYMBOL_COL,
        value_name = TARGET_COL,
    )

    print(f"[MELT] Long-format shape : {long_df.shape}")
    print(f"[MELT] Columns           : {long_df.columns.tolist()}")
    return long_df


long_df = melt_to_long(raw_wide_df)


[MELT] Long-format shape : (27300, 4)
[MELT] Columns           : ['datetime', 'underlying_price', 'symbol', 'iv']


In [ ]:
# ── 3. PARSE SYMBOL METADATA (strike, expiry, option_type) ───────────────────

# Symbol format: NIFTY<DDMMMYY><STRIKE><CE|PE>
# Example      : NIFTY27JAN2625200CE
#                └─ index ──┘└─expiry─┘└strike┘└type┘
#   index       = NIFTY  (fixed prefix, letters only)
#   expiry      = 27JAN26 → parsed to datetime
#   strike      = 25200   (integer)
#   option_type = CE | PE

_SYMBOL_RE = re.compile(
    r"^(?P<index>[A-Z]+)"           # e.g. NIFTY
    r"(?P<day>\d{2})"               # e.g. 27
    r"(?P<month>[A-Z]{3})"          # e.g. JAN
    r"(?P<year>\d{2})"              # e.g. 26
    r"(?P<strike>\d+)"              # e.g. 25200
    r"(?P<option_type>CE|PE)$"      # CE or PE
)

def parse_symbol(symbol: str) -> dict:
    """
    Extract structured fields from a single option symbol string.

    Returns a dict with keys: index, expiry, strike, option_type.
    Raises ValueError if the symbol does not match the expected pattern.
    """
    m = _SYMBOL_RE.match(symbol)
    if not m:
        raise ValueError(f"Symbol '{symbol}' does not match expected pattern.")

    expiry_str = f"{m['day']}{m['month']}{m['year']}"   # e.g. 27JAN26
    expiry     = pd.to_datetime(expiry_str, format="%d%b%y")

    return {
        "index"      : m["index"],
        "expiry"     : expiry,
        "strike"     : int(m["strike"]),
        "option_type": m["option_type"],
    }


def attach_symbol_metadata(df: pd.DataFrame) -> pd.DataFrame:
    """
    Parse all unique symbols once (not row-by-row) for efficiency,
    then merge the extracted fields back onto the long DataFrame.
    """
    unique_symbols = df[SYMBOL_COL].unique()
    print(f"[PARSE] Parsing {len(unique_symbols):,} unique symbols …")

    records = []
    failed  = []
    for sym in unique_symbols:
        try:
            parsed = parse_symbol(sym)
            parsed[SYMBOL_COL] = sym
            records.append(parsed)
        except ValueError as e:
            failed.append(str(e))

    if failed:
        print(f"[PARSE] ⚠ {len(failed)} symbols failed to parse:")
        for msg in failed[:10]:       # show first 10 only
            print(f"         {msg}")

    meta_df = pd.DataFrame(records)   # columns: symbol, index, expiry, strike, option_type
    df      = df.merge(meta_df, on=SYMBOL_COL, how="left")

    print(f"[PARSE] Metadata attached. New columns: "
          f"{['index','expiry','strike','option_type']}")
    print(f"[PARSE] Unique strikes      : {df['strike'].nunique():,}")
    print(f"[PARSE] Unique expiries     : {df['expiry'].nunique()}")
    print(f"[PARSE] Option types        : {df['option_type'].unique().tolist()}")
    return df


long_df = attach_symbol_metadata(long_df)


[PARSE] Parsing 28 unique symbols …
[PARSE] Metadata attached. New columns: ['index', 'expiry', 'strike', 'option_type']
[PARSE] Unique strikes      : 28
[PARSE] Unique expiries     : 1
[PARSE] Option types        : ['CE', 'PE']


In [ ]:
# ── 4. PARSE DATETIME & SORT ASCENDING ───────────────────────────────────────

def parse_and_sort_timestamps(df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert the datetime column to pandas Timestamp and sort the entire
    DataFrame by [datetime, strike, option_type] in ascending order.

    Attach a monotonic integer 'time_idx' derived ONLY from the observed
    ordering — no future information is encoded here.
    """
    df = df.copy()
    df[DATETIME_COL] = pd.to_datetime(df[DATETIME_COL], format='%d-%m-%Y %H:%M')
    df = df.sort_values([DATETIME_COL, "strike", "option_type"]).reset_index(drop=True)

    time_map = {ts: i for i, ts in enumerate(sorted(df[DATETIME_COL].unique()))}
    df["time_idx"] = df[DATETIME_COL].map(time_map)

    print(f"[TIME] Range           : {df[DATETIME_COL].min()} → {df[DATETIME_COL].max()}")
    print(f"[TIME] Unique timestamps: {df['time_idx'].max() + 1:,}")
    return df


long_df = parse_and_sort_timestamps(long_df)

[TIME] Range           : 2026-01-07 09:15:00 → 2026-01-27 15:25:00
[TIME] Unique timestamps: 975


In [ ]:
# ── 5. SPLIT INTO OBSERVED vs PREDICT ────────────────────────────────────────

def split_observed_missing(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Hard-split the long DataFrame into:
      - observed_df  : rows where IV is NOT NaN  → used for train/validation
      - predict_df   : rows where IV IS NaN      → submission targets

    This split must happen before any aggregation or feature engineering
    to guarantee no target leakage into training statistics.
    """
    observed   = df[df[TARGET_COL].notna()].copy()
    to_predict = df[df[TARGET_COL].isna()].copy()

    total = len(df)
    print(f"[SPLIT] Total rows     : {total:,}")
    print(f"[SPLIT] Observed       : {len(observed):,}  ({len(observed)/total*100:.1f}%)")
    print(f"[SPLIT] To predict     : {len(to_predict):,}  ({len(to_predict)/total*100:.1f}%)")
    return observed, to_predict


observed_df, predict_df = split_observed_missing(long_df)

[SPLIT] Total rows     : 27,300
[SPLIT] Observed       : 21,840  (80.0%)
[SPLIT] To predict     : 5,460  (20.0%)


In [ ]:
# ── 6. CROSS-SECTIONAL OUTLIER FLAGGING (NO LOOKAHEAD) ───────────────────────

def flag_outliers_cross_sectional(df: pd.DataFrame) -> pd.DataFrame:
    """
    For each unique datetime, compute [LOWER_QUANTILE, UPPER_QUANTILE]
    bounds on IV using ONLY the strikes observable at that timestamp.

    Flagged rows get is_outlier=True. They are kept in the dataset but
    can be down-weighted during model training.

    ✓ No-lookahead guarantee: quantile bounds are derived within each
      timestamp slice independently — no information from t+1 onward.
    """
    df = df.copy()

    def _flag(grp):
        lo = grp[TARGET_COL].quantile(LOWER_QUANTILE)
        hi = grp[TARGET_COL].quantile(UPPER_QUANTILE)
        grp["is_outlier"] = ~grp[TARGET_COL].between(lo, hi)
        return grp

    df = df.groupby(DATETIME_COL, group_keys=False).apply(_flag)

    n = df["is_outlier"].sum()
    print(f"[OUTLIER] Flagged {n:,} rows ({n/len(df)*100:.2f}%) "
          f"via cross-sectional [{LOWER_QUANTILE*100:.0f}%–{UPPER_QUANTILE*100:.0f}%] bounds.")
    return df


observed_df = flag_outliers_cross_sectional(observed_df)


[OUTLIER] Flagged 1,947 rows (8.91%) via cross-sectional [1%–99%] bounds.


In [ ]:
# ── 7. CROSS-SECTIONAL SMILE PROFILE ─────────────────────────────────────────

def compute_cross_sectional_stats(df: pd.DataFrame) -> pd.DataFrame:
    """
    For each timestamp, compute per-smile summary statistics (mean IV,
    std, min, max, count). Useful for detecting regime shifts or data
    quality issues before modelling.

    All aggregations are strictly within-timestamp — no time-axis leakage.
    """
    stats = (
        df.groupby(DATETIME_COL)[TARGET_COL]
          .agg(iv_mean="mean", iv_std="std", iv_min="min",
               iv_max="max", iv_count="count")
          .reset_index()
    )
    print(f"[STATS] Smile profile computed across {len(stats):,} timestamps.")
    print(stats.describe().round(4).to_string())
    return stats


cs_stats = compute_cross_sectional_stats(observed_df)

[STATS] Smile profile computed across 975 timestamps.
                            datetime   iv_mean    iv_std    iv_min    iv_max  iv_count
count                            975  975.0000  975.0000  975.0000  975.0000  975.0000
mean   2026-01-16 17:52:18.461538560    0.1829    0.0536    0.1125    0.2861   22.4000
min              2026-01-07 09:15:00    0.1140    0.0195    0.0168    0.1618   14.0000
25%              2026-01-12 10:47:30    0.1260    0.0278    0.0942    0.1838   21.0000
50%              2026-01-16 12:20:00    0.1339    0.0309    0.1012    0.1972   23.0000
75%              2026-01-21 13:52:30    0.1483    0.0359    0.1091    0.2234   24.0000
max              2026-01-27 15:25:00    2.6133    1.5117    0.3717    5.3848   28.0000
std                              NaN    0.1909    0.0991    0.0477    0.3579    2.0691


In [ ]:
# ── 8. SAVE CLEANED ARTEFACTS ─────────────────────────────────────────────────

observed_df.to_parquet(CLEANED_OBS_PATH, index=False)
predict_df.to_parquet(PREDICT_PATH,      index=False)
cs_stats.to_parquet(CS_STATS_PATH,       index=False)

print(f"\n[SAVE] Cleaned observed  → {CLEANED_OBS_PATH}")
print(f"[SAVE] Predict template  → {PREDICT_PATH}")
print(f"[SAVE] Smile stats       → {CS_STATS_PATH}")


[SAVE] Cleaned observed  → cleaned_observed.parquet
[SAVE] Predict template  → predict_template.parquet
[SAVE] Smile stats       → cross_sectional_stats.parquet


In [ ]:
# ── 9. SANITY CHECKS ─────────────────────────────────────────────────────────

def sanity_check(observed: pd.DataFrame, predict: pd.DataFrame) -> None:
    """
    Hard assertions before any feature engineering begins:
      1. Zero IV values in the prediction set (leakage guard).
      2. Timestamps are sorted ascending in observed set.
      3. No duplicate (datetime, symbol) pairs in observed set.
      4. All expected metadata columns are present and non-null.
    """
    # 1. Leakage guard
    leaked = predict[TARGET_COL].notna().sum()
    assert leaked == 0, \
        f"TARGET LEAKAGE: {leaked} non-null IV values found in predict set!"

    # 2. Sort order
    ts = observed[DATETIME_COL].values
    assert (ts[:-1] <= ts[1:]).all(), \
        "Timestamps are NOT sorted — recheck parse_and_sort_timestamps()."

    # 3. Duplicates
    dupes = observed.duplicated(subset=[DATETIME_COL, SYMBOL_COL]).sum()
    assert dupes == 0, \
        f"Found {dupes} duplicate (datetime, symbol) pairs in observed set!"

    # 4. Metadata completeness
    for col in ["strike", "expiry", "option_type"]:
        nulls = observed[col].isna().sum()
        assert nulls == 0, \
            f"Column '{col}' has {nulls} nulls — symbol parsing failed for some rows."

    print("\n[SANITY] ✓ All checks passed — ready for Section 2 (Feature Engineering).")


sanity_check(observed_df, predict_df)

# =============================================================================
# FINAL SCHEMA OF observed_df
# =============================================================================
print("\n[SCHEMA] observed_df columns and dtypes:")
print(observed_df.dtypes.to_string())
print(f"\n[SCHEMA] Sample rows:")
print(observed_df.head(3).to_string())

# =============================================================================
# END OF SECTION 1 (v2)
# =============================================================================


[SANITY] ✓ All checks passed — ready for Section 2 (Feature Engineering).

[SCHEMA] observed_df columns and dtypes:
datetime            datetime64[ns]
underlying_price           float64
symbol                      object
iv                         float64
index                       object
expiry              datetime64[ns]
strike                       int64
option_type                 object
time_idx                     int64
is_outlier                    bool

[SCHEMA] Sample rows:
             datetime  underlying_price               symbol       iv  index     expiry  strike option_type  time_idx  is_outlier
0 2026-01-07 09:15:00          26111.65  NIFTY27JAN2623800PE  0.17840  NIFTY 2026-01-27   23800          PE         0        True
1 2026-01-07 09:15:00          26111.65  NIFTY27JAN2623900PE  0.17237  NIFTY 2026-01-27   23900          PE         0       False
2 2026-01-07 09:15:00          26111.65  NIFTY27JAN2624000PE  0.16928  NIFTY 2026-01-27   24000          PE         0   